<a href="https://colab.research.google.com/github/judithjosephantony/Group-Project/blob/main/lyapunov_exponent_solver_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# Lorenz system
def lorenz_deriv(state):
    x, y, z = state
    sigma, rho, beta = 10.0, 28.0, 8.0/3.0
    dxdt = sigma * (y - x)
    dydt = x * (rho - z) - y
    dzdt = x * y - beta * z
    return [dxdt, dydt, dzdt]

# Rössler system
def rossler_deriv(state):
    x, y, z = state
    a, b, c = 0.2, 0.2, 5.7
    dxdt = -y - z
    dydt = x + a * y
    dzdt = b + z * (x - c)
    return [dxdt, dydt, dzdt]

In [ ]:
from scipy.integrate import solve_ivp

def get_lyapunov_exponent(deriv_func, initial_state, T=100.0, dt=0.01, method='RK45'):
    """
    Calculates the Maximal Lyapunov Exponent (MLE) using the rescaling method.
    """
    d0 = 1e-8  # Initial tiny separation
    state = np.array(initial_state)
    perturbed_state = state + np.array([d0, 0.0, 0.0])

    lyapunov_sum = 0.0
    t_values = np.arange(0, T, dt)

    fun = lambda t, y: deriv_func(y)

    for i in range(len(t_values) - 1):
        if method == 'Euler':
            state = state + dt * np.array(deriv_func(state))
            perturbed_state = perturbed_state + dt * np.array(deriv_func(perturbed_state))
        elif method == 'Predictor-Corrector':
            state_predictor = state + dt * np.array(deriv_func(state))
            perturbed_state_predictor = perturbed_state + dt * np.array(deriv_func(perturbed_state))
            state = state + dt/2 * (np.array(deriv_func(state)) + np.array(deriv_func(state_predictor)))
            perturbed_state = perturbed_state + dt/2 * (np.array(deriv_func(perturbed_state)) + np.array(deriv_func(perturbed_state_predictor)))
        elif method == 'RK4':
            k1 = np.array(deriv_func(state))
            k2 = np.array(deriv_func(state + dt/2 * k1))
            k3 = np.array(deriv_func(state + dt/2 * k2))
            k4 = np.array(deriv_func(state + dt * k3))
            state = state + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

            k1 = np.array(deriv_func(perturbed_state))
            k2 = np.array(deriv_func(perturbed_state + dt/2 * k1))
            k3 = np.array(deriv_func(perturbed_state + dt/2 * k2))
            k4 = np.array(deriv_func(perturbed_state + dt * k3))
            perturbed_state = perturbed_state + dt/6 * (k1 + 2*k2 + 2*k3 + k4)
        else:
            # Use Scipy's Runge-Kutta (RK45)
            sol_ref = solve_ivp(fun, [0, dt], state, method='RK45', rtol=1e-6, atol=1e-8)
            sol_pert = solve_ivp(fun, [0, dt], perturbed_state, method='RK45', rtol=1e-6, atol=1e-8)
            state = sol_ref.y[:, -1]
            perturbed_state = sol_pert.y[:, -1]

        # Measure Separation
        d1 = np.linalg.norm(perturbed_state - state)
        if d1 > 0:
            lyapunov_sum += np.log(d1 / d0)

        # Rescale (Renormalisation)
        diff_vector = perturbed_state - state
        perturbed_state = state + (diff_vector / d1) * d0

    mle = lyapunov_sum / T
    return mle

In [ ]:
import pandas as pd

def calculate_all_methods():
    methods = ['Euler', 'Predictor-Corrector', 'RK4', 'Scipy-RK45']

    # Expected values for comparison
    expected_lorenz = 0.90
    expected_rossler = 0.07

    # Store results
    results = []

    # Lorenz system comparison
    for method in methods:
        lorenz_lambda = get_lyapunov_exponent(lorenz_deriv, [1.0, 1.0, 1.0], T=200.0, dt=0.01, method=method)
        results.append(['Lorenz', method, lorenz_lambda, expected_lorenz, lorenz_lambda - expected_lorenz])

    # Rössler system comparison
    for method in methods:
        rossler_lambda = get_lyapunov_exponent(rossler_deriv, [1.0, 1.0, 1.0], T=500.0, dt=0.01, method=method)
        results.append(['Rössler', method, rossler_lambda, expected_rossler, rossler_lambda - expected_rossler])

    # Create DataFrame for results
    df = pd.DataFrame(results, columns=['System', 'Method', 'Calculated MLE', 'Expected MLE', 'Difference'])
    return df

In [ ]:
# Execute and display table with results
results_df = calculate_all_methods()
print(results_df)

    System               Method  Calculated MLE  Expected MLE  Difference
0   Lorenz                Euler        1.017202          0.90    0.117202
1   Lorenz  Predictor-Corrector        0.874733          0.90   -0.025267
2   Lorenz                  RK4        0.843230          0.90   -0.056770
3   Lorenz           Scipy-RK45        0.863442          0.90   -0.036558
4  Rössler                Euler        0.004793          0.07   -0.065207
5  Rössler  Predictor-Corrector        0.069971          0.07   -0.000029
6  Rössler                  RK4        0.078274          0.07    0.008274
7  Rössler           Scipy-RK45        0.072731          0.07    0.002731


Not representative of solver accuracy, so repeated with higher T and smaller dt

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import pandas as pd

# Lorenz and Rössler systems
def lorenz_deriv(state):
    x, y, z = state
    sigma, rho, beta = 10.0, 28.0, 8.0/3.0
    dxdt = sigma * (y - x)
    dydt = x * (rho - z) - y
    dzdt = x * y - beta * z
    return [dxdt, dydt, dzdt]

def rossler_deriv(state):
    x, y, z = state
    a, b, c = 0.2, 0.2, 5.7
    dxdt = -y - z
    dydt = x + a * y
    dzdt = b + z * (x - c)
    return [dxdt, dydt, dzdt]

# Improved Lyapunov Exponent Calculation
def get_lyapunov_exponent(deriv_func, initial_state, T=5000.0, dt=0.01, method='RK45'):
    """
    Calculates the Maximal Lyapunov Exponent (MLE) using the rescaling method.
    """
    d0 = 1e-8  # Initial tiny separation
    state = np.array(initial_state)
    perturbed_state = state + np.array([d0, 0.0, 0.0])

    lyapunov_sum = 0.0
    t_values = np.arange(0, T, dt)

    fun = lambda t, y: deriv_func(y)

    for i in range(len(t_values) - 1):
        if method == 'Euler':
            state = state + dt * np.array(deriv_func(state))
            perturbed_state = perturbed_state + dt * np.array(deriv_func(perturbed_state))
        elif method == 'Predictor-Corrector':
            state_predictor = state + dt * np.array(deriv_func(state))
            perturbed_state_predictor = perturbed_state + dt * np.array(deriv_func(perturbed_state))
            state = state + dt/2 * (np.array(deriv_func(state)) + np.array(deriv_func(state_predictor)))
            perturbed_state = perturbed_state + dt/2 * (np.array(deriv_func(perturbed_state)) + np.array(deriv_func(perturbed_state_predictor)))
        elif method == 'RK4':
            k1 = np.array(deriv_func(state))
            k2 = np.array(deriv_func(state + dt/2 * k1))
            k3 = np.array(deriv_func(state + dt/2 * k2))
            k4 = np.array(deriv_func(state + dt * k3))
            state = state + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

            k1 = np.array(deriv_func(perturbed_state))
            k2 = np.array(deriv_func(perturbed_state + dt/2 * k1))
            k3 = np.array(deriv_func(perturbed_state + dt/2 * k2))
            k4 = np.array(deriv_func(perturbed_state + dt * k3))
            perturbed_state = perturbed_state + dt/6 * (k1 + 2*k2 + 2*k3 + k4)
        else:
            # Use Scipy's Runge-Kutta (RK45)
            sol_ref = solve_ivp(fun, [0, dt], state, method='RK45', rtol=1e-6, atol=1e-8)
            sol_pert = solve_ivp(fun, [0, dt], perturbed_state, method='RK45', rtol=1e-6, atol=1e-8)
            state = sol_ref.y[:, -1]
            perturbed_state = sol_pert.y[:, -1]

        # Measure Separation
        d1 = np.linalg.norm(perturbed_state - state)
        if d1 > 0:
            lyapunov_sum += np.log(d1 / d0)

        # Rescale (Renormalisation)
        diff_vector = perturbed_state - state
        perturbed_state = state + (diff_vector / d1) * d0

    mle = lyapunov_sum / T
    return mle

# Run the comparisons using improved methods
def calculate_all_methods():
    methods = ['Euler', 'Predictor-Corrector', 'RK4', 'Scipy-RK45']

    # Expected values for comparison
    expected_lorenz = 0.90
    expected_rossler = 0.07

    results = []

    # Lorenz system comparison
    for method in methods:
        lorenz_lambda = get_lyapunov_exponent(lorenz_deriv, [1.0, 1.0, 1.0], T=5000.0, dt=0.01, method=method)
        results.append(['Lorenz', method, lorenz_lambda, expected_lorenz, lorenz_lambda - expected_lorenz])

    # Rössler system comparison
    for method in methods:
        rossler_lambda = get_lyapunov_exponent(rossler_deriv, [1.0, 1.0, 1.0], T=5000.0, dt=0.01, method=method)
        results.append(['Rössler', method, rossler_lambda, expected_rossler, rossler_lambda - expected_rossler])

    # Create DataFrame for results
    df = pd.DataFrame(results, columns=['System', 'Method', 'Calculated MLE', 'Expected MLE', 'Difference'])
    return df

# Execute and display table with results
results_df = calculate_all_methods()
print(results_df)

    System               Method  Calculated MLE  Expected MLE  Difference
0   Lorenz                Euler        1.038997          0.90    0.138997
1   Lorenz  Predictor-Corrector        0.907828          0.90    0.007828
2   Lorenz                  RK4        0.904397          0.90    0.004397
3   Lorenz           Scipy-RK45        0.907071          0.90    0.007071
4  Rössler                Euler        0.000312          0.07   -0.069688
5  Rössler  Predictor-Corrector        0.071386          0.07    0.001386
6  Rössler                  RK4        0.074023          0.07    0.004023
7  Rössler           Scipy-RK45        0.071758          0.07    0.001758


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import pandas as pd

# Lorenz and Rössler systems
def lorenz_deriv(state):
    x, y, z = state
    sigma, rho, beta = 10.0, 28.0, 8.0/3.0
    dxdt = sigma * (y - x)
    dydt = x * (rho - z) - y
    dzdt = x * y - beta * z
    return [dxdt, dydt, dzdt]

def rossler_deriv(state):
    x, y, z = state
    a, b, c = 0.2, 0.2, 5.7
    dxdt = -y - z
    dydt = x + a * y
    dzdt = b + z * (x - c)
    return [dxdt, dydt, dzdt]

# Improved Lyapunov Exponent Calculation
def get_lyapunov_exponent(deriv_func, initial_state, T=10000.0, dt=0.01, method='RK45'):
    """
    Calculates the Maximal Lyapunov Exponent (MLE) using the rescaling method.
    """
    d0 = 1e-8  # Initial tiny separation
    state = np.array(initial_state)
    perturbed_state = state + np.array([d0, 0.0, 0.0])

    lyapunov_sum = 0.0
    t_values = np.arange(0, T, dt)

    fun = lambda t, y: deriv_func(y)

    for i in range(len(t_values) - 1):
        if method == 'Euler':
            state = state + dt * np.array(deriv_func(state))
            perturbed_state = perturbed_state + dt * np.array(deriv_func(perturbed_state))
        elif method == 'Predictor-Corrector':
            state_predictor = state + dt * np.array(deriv_func(state))
            perturbed_state_predictor = perturbed_state + dt * np.array(deriv_func(perturbed_state))
            state = state + dt/2 * (np.array(deriv_func(state)) + np.array(deriv_func(state_predictor)))
            perturbed_state = perturbed_state + dt/2 * (np.array(deriv_func(perturbed_state)) + np.array(deriv_func(perturbed_state_predictor)))
        elif method == 'RK4':
            k1 = np.array(deriv_func(state))
            k2 = np.array(deriv_func(state + dt/2 * k1))
            k3 = np.array(deriv_func(state + dt/2 * k2))
            k4 = np.array(deriv_func(state + dt * k3))
            state = state + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

            k1 = np.array(deriv_func(perturbed_state))
            k2 = np.array(deriv_func(perturbed_state + dt/2 * k1))
            k3 = np.array(deriv_func(perturbed_state + dt/2 * k2))
            k4 = np.array(deriv_func(perturbed_state + dt * k3))
            perturbed_state = perturbed_state + dt/6 * (k1 + 2*k2 + 2*k3 + k4)
        else:
            # Use Scipy's Runge-Kutta (RK45)
            sol_ref = solve_ivp(fun, [0, dt], state, method='RK45', rtol=1e-6, atol=1e-8)
            sol_pert = solve_ivp(fun, [0, dt], perturbed_state, method='RK45', rtol=1e-6, atol=1e-8)
            state = sol_ref.y[:, -1]
            perturbed_state = sol_pert.y[:, -1]

        # Measure Separation
        d1 = np.linalg.norm(perturbed_state - state)
        if d1 > 0:
            lyapunov_sum += np.log(d1 / d0)

        # Rescale (Renormalisation)
        diff_vector = perturbed_state - state
        perturbed_state = state + (diff_vector / d1) * d0

    mle = lyapunov_sum / T
    return mle

# Run the comparisons using improved methods
def calculate_all_methods():
    methods = ['Euler', 'Predictor-Corrector', 'RK4', 'Scipy-RK45']

    # Expected values for comparison
    expected_lorenz = 0.90
    expected_rossler = 0.07

    results = []

    # Lorenz system comparison
    for method in methods:
        lorenz_lambda = get_lyapunov_exponent(lorenz_deriv, [1.0, 1.0, 1.0], T=10000.0, dt=0.01, method=method)
        results.append(['Lorenz', method, lorenz_lambda, expected_lorenz, lorenz_lambda - expected_lorenz])

    # Rössler system comparison
    for method in methods:
        rossler_lambda = get_lyapunov_exponent(rossler_deriv, [1.0, 1.0, 1.0], T=10000.0, dt=0.01, method=method)
        results.append(['Rössler', method, rossler_lambda, expected_rossler, rossler_lambda - expected_rossler])

    # Create DataFrame for results
    df = pd.DataFrame(results, columns=['System', 'Method', 'Calculated MLE', 'Expected MLE', 'Difference'])
    return df

# Execute and display table with results
results_df = calculate_all_methods()
print(results_df)

    System               Method  Calculated MLE  Expected MLE  Difference
0   Lorenz                Euler        1.037828          0.90    0.137828
1   Lorenz  Predictor-Corrector        0.908652          0.90    0.008652
2   Lorenz                  RK4        0.904466          0.90    0.004466
3   Lorenz           Scipy-RK45        0.906076          0.90    0.006076
4  Rössler                Euler        0.000238          0.07   -0.069762
5  Rössler  Predictor-Corrector        0.070961          0.07    0.000961
6  Rössler                  RK4        0.072796          0.07    0.002796
7  Rössler           Scipy-RK45        0.071039          0.07    0.001039


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import pandas as pd

# Lorenz and Rössler systems
def lorenz_deriv(state):
    x, y, z = state
    sigma, rho, beta = 10.0, 28.0, 8.0/3.0
    dxdt = sigma * (y - x)
    dydt = x * (rho - z) - y
    dzdt = x * y - beta * z
    return [dxdt, dydt, dzdt]

def rossler_deriv(state):
    x, y, z = state
    a, b, c = 0.2, 0.2, 5.7
    dxdt = -y - z
    dydt = x + a * y
    dzdt = b + z * (x - c)
    return [dxdt, dydt, dzdt]

# Improved Lyapunov Exponent Calculation
def get_lyapunov_exponent(deriv_func, initial_state, T=10000.0, dt=0.005, method='RK45'):
    """
    Calculates the Maximal Lyapunov Exponent (MLE) using the rescaling method.
    """
    d0 = 1e-8  # Initial tiny separation
    state = np.array(initial_state)
    perturbed_state = state + np.array([d0, 0.0, 0.0])

    lyapunov_sum = 0.0
    t_values = np.arange(0, T, dt)

    fun = lambda t, y: deriv_func(y)

    for i in range(len(t_values) - 1):
        if method == 'Euler':
            state = state + dt * np.array(deriv_func(state))
            perturbed_state = perturbed_state + dt * np.array(deriv_func(perturbed_state))
        elif method == 'Predictor-Corrector':
            state_predictor = state + dt * np.array(deriv_func(state))
            perturbed_state_predictor = perturbed_state + dt * np.array(deriv_func(perturbed_state))
            state = state + dt/2 * (np.array(deriv_func(state)) + np.array(deriv_func(state_predictor)))
            perturbed_state = perturbed_state + dt/2 * (np.array(deriv_func(perturbed_state)) + np.array(deriv_func(perturbed_state_predictor)))
        elif method == 'RK4':
            k1 = np.array(deriv_func(state))
            k2 = np.array(deriv_func(state + dt/2 * k1))
            k3 = np.array(deriv_func(state + dt/2 * k2))
            k4 = np.array(deriv_func(state + dt * k3))
            state = state + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

            k1 = np.array(deriv_func(perturbed_state))
            k2 = np.array(deriv_func(perturbed_state + dt/2 * k1))
            k3 = np.array(deriv_func(perturbed_state + dt/2 * k2))
            k4 = np.array(deriv_func(perturbed_state + dt * k3))
            perturbed_state = perturbed_state + dt/6 * (k1 + 2*k2 + 2*k3 + k4)
        else:
            # Use Scipy's Runge-Kutta (RK45)
            sol_ref = solve_ivp(fun, [0, dt], state, method='RK45', rtol=1e-6, atol=1e-8)
            sol_pert = solve_ivp(fun, [0, dt], perturbed_state, method='RK45', rtol=1e-6, atol=1e-8)
            state = sol_ref.y[:, -1]
            perturbed_state = sol_pert.y[:, -1]

        # Measure Separation
        d1 = np.linalg.norm(perturbed_state - state)
        if d1 > 0:
            lyapunov_sum += np.log(d1 / d0)

        # Rescale (Renormalisation)
        diff_vector = perturbed_state - state
        perturbed_state = state + (diff_vector / d1) * d0

    mle = lyapunov_sum / T
    return mle

# Run the comparisons using improved methods
def calculate_all_methods():
    methods = ['Euler', 'Predictor-Corrector', 'RK4', 'Scipy-RK45']

    # Expected values for comparison
    expected_lorenz = 0.90
    expected_rossler = 0.07

    results = []

    # Lorenz system comparison
    for method in methods:
        lorenz_lambda = get_lyapunov_exponent(lorenz_deriv, [1.0, 1.0, 1.0], T=10000.0, dt=0.005, method=method)
        results.append(['Lorenz', method, lorenz_lambda, expected_lorenz, lorenz_lambda - expected_lorenz])

    # Rössler system comparison
    for method in methods:
        rossler_lambda = get_lyapunov_exponent(rossler_deriv, [1.0, 1.0, 1.0], T=10000.0, dt=0.005, method=method)
        results.append(['Rössler', method, rossler_lambda, expected_rossler, rossler_lambda - expected_rossler])

    # Create DataFrame for results
    df = pd.DataFrame(results, columns=['System', 'Method', 'Calculated MLE', 'Expected MLE', 'Difference'])
    return df

# Execute and display table with results
results_df = calculate_all_methods()
print(results_df)

    System               Method  Calculated MLE  Expected MLE  Difference
0   Lorenz                Euler        0.957376          0.90    0.057376
1   Lorenz  Predictor-Corrector        0.903238          0.90    0.003238
2   Lorenz                  RK4        0.904868          0.90    0.004868
3   Lorenz           Scipy-RK45        0.903897          0.90    0.003897
4  Rössler                Euler        0.052280          0.07   -0.017720
5  Rössler  Predictor-Corrector        0.073656          0.07    0.003656
6  Rössler                  RK4        0.071292          0.07    0.001292
7  Rössler           Scipy-RK45        0.071633          0.07    0.001633
